# Player Pair Retrieval Workflow

This notebook runs the retrieval version of the chess style project.

Instead of asking "which player is this game?", retrieval asks:

```text
Are games by the same player closer in embedding space than games by different players?
```

This is closer to the behavioral stylometry paper than direct softmax classification.

## 1. Set Project Root

This cell makes the notebook work whether you open it from the project root or from the `retrieval/` folder.

In [1]:
from pathlib import Path
import os
import sys

cwd = Path.cwd()
if cwd.name == 'retrieval':
    os.chdir(cwd.parent)
elif (cwd / 'chess-style-fingerprint').exists():
    os.chdir(cwd / 'chess-style-fingerprint')

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Has src:', (PROJECT_ROOT / 'src').exists())
print('Has data/raw:', (PROJECT_ROOT / 'data/raw').exists())

Project root: /Users/aadil/Chess-ML-Project/chess-style-fingerprint
Has src: True
Has data/raw: True


## 2. Verify Dependencies and Device

The retrieval workflow uses PyTorch, python-chess, and the local project modules.

In [2]:
import torch
import chess

print('torch:', torch.__version__)
print('python-chess:', chess.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Running on CPU')

torch: 2.11.0
python-chess: 1.11.2
CUDA available: False
Running on CPU


## 3. Inspect Available Games

This counts usable games per active label after filtering out games that are too short to produce windows.

In [4]:
from collections import Counter
from src.dataset import LABELS, collect_game_records

records = collect_game_records('data/raw')
counts = Counter(record.label_name for record in records)

print('Labels:', LABELS)
print('Usable games:')
for name in sorted(counts):
    print(f'{name:12s}: {counts[name]}')

Labels: {'tal': 0, 'petrosian': 1, 'karpov': 2, 'fischer': 3, 'kasparov': 4, 'carlsen': 5, 'anand': 6, 'botvinnik': 7, 'stockfish': 8}
Usable games:
anand       : 3626
botvinnik   : 774
carlsen     : 7046
fischer     : 729
karpov      : 2964
kasparov    : 1820
petrosian   : 1316
stockfish   : 998
tal         : 1801


## 4. Retrieval Metrics

The retrieval script reports four quantities:

- **Nearest-neighbor accuracy:** for each game, whether the closest other game embedding has the same player label.
- **Mean same-player cosine:** average similarity between games from the same player.
- **Mean different-player cosine:** average similarity between games from different players.
- **Sampled pair AUC:** probability that a random same-player pair is more similar than a random different-player pair.

For useful embeddings, same-player similarity should be higher than different-player similarity, and pair AUC should be above 50%.

## 5. Untrained Retrieval Baseline

This run uses random/untrained ModelC embeddings. It should be near random. This is a sanity check that tells us what "no learned style signal" looks like.

In [5]:
!python retrieval/run_retrieval.py   --data-dir data/raw   --max-games-per-label 10   --max-windows-per-game 4   --model-scale tiny   --exclude-label stockfish   --pair-samples 1000


Labels with no usable games:
  stockfish
Using 10 games for anand
Using 10 games for botvinnik
Using 10 games for carlsen
Using 10 games for fischer
Using 10 games for karpov
Using 10 games for kasparov
Using 10 games for petrosian
Using 10 games for tal
Game counts:
  anand       : 10
  botvinnik   : 10
  carlsen     : 10
  fischer     : 10
  karpov      : 10
  kasparov    : 10
  petrosian   : 10
  tal         : 10
Precomputing game windows...
  precomputed 50/80 games
  precomputed 80/80 games
/Users/aadil/Chess-ML-Project/chess-style-fingerprint/src/model.py:177: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
No checkpoint provided: evaluating random/untrained embeddings.

Retrieval results:
  games: 80
  labels: ['tal', 'petrosian', 'karpov', 'fischer', 'kasparov', 'carlsen', 'anand', 'botvinnik']
  nearest-neighbor accuracy: 8.75

## 6. Train a Pairwise Checkpoint

Retrieval needs meaningful embeddings. One practical way to get them quickly is to train ModelC on a pairwise classification task, then reuse its embedding layer.

This cell trains Tal vs Petrosian with the small model. You can skip it if you already have `outputs/final_tal_petrosian_small.pt`.

In [ ]:
!python scripts/run_game_embedding_experiment.py   --data-dir data/raw   --epochs 10   --steps-per-epoch 80   --batch-games 4   --max-games-per-label 100   --max-windows-per-game 10   --model-scale small   --exclude-label karpov   --exclude-label fischer   --exclude-label kasparov   --exclude-label carlsen   --exclude-label anand   --exclude-label botvinnik   --exclude-label stockfish   --save-path outputs/retrieval_tal_petrosian_small.pt

## 7. Retrieval With the Trained Pairwise Checkpoint

This evaluates whether the trained Tal/Petrosian model embeds same-player games closer together.

In [ ]:
!python retrieval/run_retrieval.py   --data-dir data/raw   --max-games-per-label 50   --max-windows-per-game 10   --model-scale small   --exclude-label karpov   --exclude-label fischer   --exclude-label kasparov   --exclude-label carlsen   --exclude-label anand   --exclude-label botvinnik   --exclude-label stockfish   --checkpoint outputs/retrieval_tal_petrosian_small.pt   --pair-samples 5000

## 8. Human-vs-Stockfish Retrieval Control

This checks whether embeddings separate human games from engine games. Treat this as a control: it tests human-vs-engine separation, not fine-grained human style.

In [ ]:
# First train a Carlsen-vs-Stockfish checkpoint.
!python scripts/run_game_embedding_experiment.py   --data-dir data/raw   --epochs 8   --steps-per-epoch 60   --batch-games 4   --max-games-per-label 100   --max-windows-per-game 8   --model-scale small   --exclude-label tal   --exclude-label petrosian   --exclude-label karpov   --exclude-label fischer   --exclude-label kasparov   --exclude-label anand   --exclude-label botvinnik   --save-path outputs/retrieval_carlsen_stockfish_small.pt

In [ ]:
# Then evaluate retrieval on Carlsen vs Stockfish.
!python retrieval/run_retrieval.py   --data-dir data/raw   --max-games-per-label 50   --max-windows-per-game 8   --model-scale small   --exclude-label tal   --exclude-label petrosian   --exclude-label karpov   --exclude-label fischer   --exclude-label kasparov   --exclude-label anand   --exclude-label botvinnik   --checkpoint outputs/retrieval_carlsen_stockfish_small.pt   --pair-samples 5000

## 9. Optional: Multi-Player Retrieval

This uses a checkpoint trained on some task and evaluates retrieval across multiple human players. Results may be weak unless the checkpoint was trained with a retrieval or contrastive objective.

In [ ]:
# Example using an existing checkpoint, if available.
# !python retrieval/run_retrieval.py \
#   --data-dir data/raw \
#   --max-games-per-label 20 \
#   --max-windows-per-game 6 \
#   --model-scale small \
#   --exclude-label stockfish \
#   --checkpoint outputs/retrieval_tal_petrosian_small.pt \
#   --pair-samples 5000


## 10. Report Notes

Good retrieval evidence looks like:

```text
nearest-neighbor accuracy above random
same-player cosine > different-player cosine
sampled pair AUC > 50%
```

If retrieval works for human-vs-Stockfish but not human-human pairs, frame the result as:

```text
The embedding captures coarse human-vs-engine decision differences, but fine-grained human player identity remains difficult without metric-learning training.
```